# 08 - Feature Exploration and Leakage Detection
Standalone notebook that reads the saved feature matrix and performs sanity checks.
**Purpose:** Verify data quality and detect look-ahead leakage before Phase 3 modeling.

Checks:
1. Feature coverage per season (% non-null)
2. Feature-outcome correlation (flag > 0.7 as potential leakage)
3. Temporal ordering spot-check (verify shift(1) working)

In [ ]:
import sys
sys.path.insert(0, "..")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Read saved feature matrix (does NOT re-run FeatureBuilder)
df = pd.read_parquet("../data/features/feature_matrix.parquet")
print(f"Feature matrix: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Seasons: {sorted(df['season'].unique())}")

## 1. Feature Coverage Per Season
% non-null for each feature column per season. Expect:
- SP features: high coverage (most games have confirmed starters)
- Rolling features: ~94% (games 1-9 of each season are NaN)
- SP recent form (30-day ERA): may have lower coverage in early season (short window)
- Kalshi: 0% for 2015-2024, partial for 2025
- xwOBA: may have gaps for some seasons

In [ ]:
# Identify feature columns (exclude metadata)
meta_cols = ['game_date', 'home_team', 'away_team', 'season', 'home_win',
             'home_probable_pitcher', 'away_probable_pitcher',
             'is_shortened_season', 'game_id', 'status',
             'home_score', 'away_score', 'winning_team', 'losing_team']
feature_cols = [c for c in df.columns if c not in meta_cols]

# Coverage table: % non-null per feature per season
coverage = df.groupby('season')[feature_cols].apply(lambda x: x.notna().mean() * 100)
print("Feature Coverage (% non-null per season):")
print(coverage.round(1).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(coverage.T, annot=True, fmt='.0f', cmap='RdYlGn', vmin=0, vmax=100,
            ax=ax, cbar_kws={'label': '% non-null'})
ax.set_title('Feature Coverage by Season (% non-null)')
ax.set_xlabel('Season')
ax.set_ylabel('Feature')
plt.tight_layout()
plt.show()

## 2. Feature-Outcome Correlation
Pearson correlation of each feature with `home_win`.
- In sports prediction, 0.3 is a legitimately strong signal.
- Flag > 0.7 as potential leakage (suspiciously high for a noisy domain).
- Flag > 0.5 for manual review.

In [ ]:
# Compute correlation with home_win for numeric feature columns
numeric_features = df[feature_cols].select_dtypes(include=[np.number]).columns
correlations = df[list(numeric_features)].corrwith(df['home_win']).sort_values(ascending=False)

print("Feature correlations with home_win:")
print("=" * 50)
for feat, corr in correlations.items():
    flag = ""
    if abs(corr) > 0.7:
        flag = " *** LEAKAGE ALERT ***"
    elif abs(corr) > 0.5:
        flag = " ** HIGH (review) **"
    elif abs(corr) > 0.3:
        flag = " * strong signal *"
    print(f"  {feat:30s}  {corr:+.4f}{flag}")

# Automated leakage check
leakage_features = [f for f, c in correlations.items() if abs(c) > 0.7]
if leakage_features:
    print(f"\n!!! POTENTIAL LEAKAGE DETECTED in: {leakage_features}")
    print("Investigate these features before proceeding to Phase 3!")
else:
    print(f"\nNo features with |correlation| > 0.7. Leakage check PASSED.")

In [ ]:
# Correlation matrix heatmap for all numeric features
corr_matrix = df[list(numeric_features)].corr()
fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, ax=ax,
            square=True, linewidths=0.5)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 3. Temporal Ordering Spot-Check
Verify that `shift(1)` is working correctly:
- Pick a mid-season game
- Confirm its rolling features are computed from PRIOR games only
- Confirm removing that game's outcome would not change its feature values

In [ ]:
# Pick a mid-season game from 2023
season_2023 = df[df['season'] == 2023].sort_values('game_date')
mid_game = season_2023.iloc[len(season_2023) // 2]

print(f"Spot-check game: {mid_game['home_team']} vs {mid_game['away_team']} on {mid_game['game_date']}")
print(f"  home_win: {mid_game['home_win']}")
print(f"  rolling_ops_diff: {mid_game.get('rolling_ops_diff', 'N/A')}")
print(f"  log5_home_wp: {mid_game.get('log5_home_wp', 'N/A')}")
print(f"  sp_recent_era_diff: {mid_game.get('sp_recent_era_diff', 'N/A')}")

# Check early-season NaN behavior
early_games = season_2023.head(15)
rolling_col = [c for c in df.columns if 'rolling' in c.lower()]
if rolling_col:
    rc = rolling_col[0]
    early_nan_count = early_games[rc].isna().sum()
    print(f"\nEarly-season NaN check ({rc}):")
    print(f"  First 15 games: {early_nan_count} NaN values")
    print(f"  Expected: 9-10 NaN (games 1-9 or 1-10 due to shift+min_periods)")
    if early_nan_count >= 9:
        print("  Temporal safety check PASSED")
    else:
        print("  !!! WARNING: Fewer NaN than expected. Check shift(1) implementation.")

In [ ]:
# Distribution of key differential features
fig, axes = plt.subplots(4, 3, figsize=(15, 16))
plot_features = ['sp_fip_diff', 'team_ops_diff', 'rolling_ops_diff',
                 'bullpen_era_diff', 'park_factor', 'log5_home_wp',
                 'sp_siera_diff', 'xwoba_diff', 'bullpen_fip_diff',
                 'sp_recent_era_diff', 'pyth_win_pct_diff', 'team_woba_diff']
for ax, feat in zip(axes.flat, plot_features):
    if feat in df.columns:
        df[feat].dropna().hist(bins=50, ax=ax, alpha=0.7)
        ax.set_title(feat)
        ax.set_xlabel('')
    else:
        ax.set_title(f'{feat} (missing)')
plt.suptitle('Feature Distributions', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 60)
print("FEATURE MATRIX SUMMARY")
print("=" * 60)
print(f"Total games: {len(df)}")
print(f"Seasons: {sorted(df['season'].unique())}")
print(f"Home win rate: {df['home_win'].mean():.3f}")
print(f"Feature columns: {len(feature_cols)}")
print(f"Kalshi coverage: {df['kalshi_yes_price'].notna().sum()} games")
print(f"SP recent form coverage: {df['sp_recent_era_diff'].notna().sum()} games with 30-day ERA")
print(f"2020 short-season games: {df['is_shortened_season'].sum()}")
leakage_count = len([f for f, c in correlations.items() if abs(c) > 0.7])
print(f"Leakage alerts: {leakage_count}")
print(f"\nReady for Phase 3: {'YES' if leakage_count == 0 else 'NO - investigate leakage first'}")